# Лабораторна робота №1
## «Зменшення розмірності даних»

**Мета:** Ознайомитися з основами зменшення розмірності даних на прикладі зображень рукописних цифр із набору MNIST. Навчитись виконувати метод головних компонент (PCA), інтерпретувати результати, оцінювати втрату інформації після проєкції та реконструкції зображень.

## 0. Імпорт бібліотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print("Всі бібліотеки успішно імпортовано!")

## 1. Завантаження та підготовка даних

In [ ]:
# Завантаження датасету MNIST
print("Завантаження MNIST... (може зайняти кілька хвилин)")
X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)

# Перетворення міток у числа
y = y.astype(int)

print(f"Форма матриці X: {X.shape}")
print(f"Форма вектора y: {y.shape}")
print(f"Унікальні мітки (класи): {np.unique(y)}")
print(f"Кількість унікальних міток: {len(np.unique(y))}")

In [ ]:
# Нормалізація даних (приведення пікселів до діапазону [0, 1])
X_norm = X / 255.0

# Візуалізація 10 випадкових зображень
np.random.seed(42)
random_indices = np.random.choice(len(X), size=10, replace=False)

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
fig.suptitle('Приклади зображень з датасету MNIST', fontsize=14)

for i, idx in enumerate(random_indices):
    axes[i].imshow(X[idx].reshape(28, 28), cmap='gray')
    axes[i].set_title(f'Цифра: {y[idx]}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print(f"Мінімальне значення пікселя: {X.min()}, Максимальне: {X.max()}")

## 2. Реалізація методу головних компонент (PCA)

In [ ]:
# Застосування PCA з 3 компонентами
pca_3 = PCA(n_components=3, random_state=42)
X_pca = pca_3.fit_transform(X_norm)

print(f"Форма матриці після PCA (X_pca): {X_pca.shape}")
print()
print("Відсоток поясненої дисперсії для кожної компоненти:")
for i, var in enumerate(pca_3.explained_variance_ratio_):
    print(f"  PC{i+1}: {var:.4f} ({var*100:.2f}%)")

total_var = sum(pca_3.explained_variance_ratio_)
print(f"\nЗагальна пояснена дисперсія (3 компоненти): {total_var:.4f} ({total_var*100:.2f}%)")

## 3. Візуалізація даних у просторі перших трьох компонент

In [ ]:
# Для кращої читабельності використаємо підвибірку
sample_size = 5000
np.random.seed(42)
sample_idx = np.random.choice(len(X_pca), size=sample_size, replace=False)

X_sample = X_pca[sample_idx]
y_sample = y[sample_idx]

# 3D-графік
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

colors = plt.cm.tab10(np.linspace(0, 1, 10))

for digit in range(10):
    mask = y_sample == digit
    ax.scatter(
        X_sample[mask, 0],
        X_sample[mask, 1],
        X_sample[mask, 2],
        c=[colors[digit]],
        label=str(digit),
        alpha=0.5,
        s=5
    )

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('3D-візуалізація MNIST у просторі PCA (PC1, PC2, PC3)')
ax.legend(title='Цифра', loc='upper left', markerscale=3)

plt.tight_layout()
plt.show()

In [ ]:
# Аналіз кластеризації
print("Аналіз кластеризації у просторі PCA:")
print()
print("З 3D-графіку видно, що:")
print(" - Деякі цифри (наприклад, 0 та 1) формують відносно відокремлені кластери.")
print(" - Більшість класів перекриваються у просторі трьох компонент.")
print(" - Це пояснюється тим, що 3 компоненти пояснюють лише ~\"N%\" дисперсії.")
print(f" - Фактично, перші 3 PC пояснюють {total_var*100:.1f}% загальної дисперсії.")
print()
print("Висновок: PCA з 3 компонентами не забезпечує чіткого розділення всіх класів.")
print("Для кращого розділення потрібна більша кількість компонент.")

## 4. Реконструкція зображень після зменшення розмірності

In [ ]:
# Реконструкція через inverse_transform
X_reconstructed = pca_3.inverse_transform(X_pca)

print(f"Форма реконструйованих даних: {X_reconstructed.shape}")

# Візуалізація оригінальних та реконструйованих зображень
np.random.seed(42)
indices = np.random.choice(len(X_norm), size=10, replace=False)

fig, axes = plt.subplots(2, 10, figsize=(18, 4))
fig.suptitle('Порівняння оригінальних та реконструйованих зображень (PCA, 3 компоненти)', fontsize=13)

for i, idx in enumerate(indices):
    # Оригінальне зображення
    axes[0, i].imshow(X_norm[idx].reshape(28, 28), cmap='gray')
    axes[0, i].set_title(f'{y[idx]}', fontsize=10)
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Оригінал', fontsize=10)

    # Реконструйоване зображення
    axes[1, i].imshow(X_reconstructed[idx].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Рекон-ція', fontsize=10)

axes[0, 0].set_ylabel('Оригінал', fontsize=11)
axes[1, 0].set_ylabel('Реконст.\n(PCA, k=3)', fontsize=11)

plt.tight_layout()
plt.show()

# MSE для 3 компонент
mse_3 = mean_squared_error(X_norm, X_reconstructed)
print(f"\nMSE при 3 компонентах: {mse_3:.6f}")

## 5. Аналіз залежності MSE від кількості компонент

In [ ]:
# Список значень k для аналізу
k_values = [1, 2, 3, 5, 10, 20, 30, 50, 75, 100, 150, 200]

mse_list = []
variance_list = []

print("Обчислення MSE та поясненої дисперсії для різних k...")

for k in k_values:
    pca_k = PCA(n_components=k, random_state=42)
    X_proj = pca_k.fit_transform(X_norm)
    X_rec = pca_k.inverse_transform(X_proj)
    
    mse = mean_squared_error(X_norm, X_rec)
    var = np.sum(pca_k.explained_variance_ratio_)
    
    mse_list.append(mse)
    variance_list.append(var * 100)
    
    print(f"k = {k:4d} | MSE = {mse:.6f} | Пояснена дисперсія = {var*100:.2f}%")

In [ ]:
# Графіки залежності MSE та поясненої дисперсії від k
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Графік MSE
ax1.plot(k_values, mse_list, 'b-o', linewidth=2, markersize=6)
ax1.axvline(x=3, color='red', linestyle='--', label='k=3 (поточне значення)')
ax1.set_xlabel('Кількість компонент (k)', fontsize=12)
ax1.set_ylabel('MSE (середня квадратична похибка)', fontsize=12)
ax1.set_title('Залежність MSE від кількості компонент PCA', fontsize=13)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log')

# Графік поясненої дисперсії
ax2.plot(k_values, variance_list, 'g-o', linewidth=2, markersize=6)
ax2.axvline(x=3, color='red', linestyle='--', label='k=3 (поточне значення)')
ax2.axhline(y=95, color='orange', linestyle='--', alpha=0.7, label='95% поясненої дисперсії')
ax2.set_xlabel('Кількість компонент (k)', fontsize=12)
ax2.set_ylabel('Пояснена дисперсія (%)', fontsize=12)
ax2.set_title('Залежність поясненої дисперсії від кількості компонент', fontsize=13)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Зведена таблиця результатів
results_df = pd.DataFrame({
    'Кількість компонент (k)': k_values,
    'MSE': [round(m, 6) for m in mse_list],
    'Пояснена дисперсія (%)': [round(v, 2) for v in variance_list]
})

print("Зведена таблиця результатів:")
print(results_df.to_string(index=False))

## 6. Висновки

**1. Завантаження та підготовка даних:**
- Датасет MNIST містить 70 000 зображень розміром 28×28 пікселів (784 ознаки), що належать до 10 класів (цифри 0–9).

**2. PCA з 3 компонентами:**
- Перші три головні компоненти пояснюють лише близько 15–20% загальної дисперсії.
- Це свідчить про те, що три компоненти несуть лише малу частину інформації з оригінального 784-вимірного простору.

**3. 3D-візуалізація:**
- У просторі трьох компонент деякі цифри (наприклад, 0 та 1) формують відносно відокремлені кластери.
- Більшість класів перекриваються, що ускладнює лінійну класифікацію в цьому просторі.

**4. Реконструкція зображень:**
- При реконструкції з 3 компонент зображення виглядають розмитими — деталі втрачаються.
- MSE при k=3 є досить великою, що підтверджує значну втрату інформації.

**5. Залежність MSE від k:**
- Зі збільшенням кількості компонент MSE монотонно зменшується.
- Для досягнення 95% поясненої дисперсії необхідно приблизно 150–200 компонент.
- Компроміс між якістю реконструкції та розмірністю даних є ключовим при виборі k.